# Context-aware agglomerative clustering

This notebook groups product attributes—short labels such as a product feature title—when they appear to describe the same property. Related titles are not always worded alike, so the workflow combines sentence-embedding meaning, informative words in descriptions, and title wording. For example, the source data includes the title `DWR coating` with descriptions about water repellency. The title alone is brief; its descriptions provide the context used by the semantic and IDF signals. A configurable business rule can discourage explicitly conflicting concepts before average-linkage agglomerative clustering.

The current preprocessed data contains 1,078 title entries reduced to 445 unique attributes, alongside 1,064 description entries. The aligned records connect 1,042 descriptions to their attributes; 22 descriptions from mismatched source blocks remain unpaired and are used for IDF vocabulary weights only. Twelve unique attributes have no confirmed aligned description.

## Problem introduction

_Write a short introduction here: what problem needs to be solved, why grouping these attributes is useful, and what the intended output will help someone do._

---

## How to read this notebook

A **similarity** near 1 means two attributes look alike under a signal; a **distance** near 0 means they are close. Agglomerative clustering starts with each attribute alone and repeatedly joins close groups until the distance cutoff is reached. The cluster labels are the algorithm's output; t-SNE is only a final picture of those labels, not another clustering method.

## Workflow
1. Load `features_en.json` created by the preprocessing notebook. The file must contain `attribute_records`, which preserves each unique title with its correctly aligned descriptions.
2. Build a TF-IDF similarity matrix from aligned descriptions; use additional unpaired descriptions only to estimate IDF weights.
3. Build one semantic vector per attribute by blending its title embedding with the frequency-weighted mean of its aligned description embeddings.
4. Blend the semantic, description-IDF, and title-string similarities, apply the example business constraint, and cluster the resulting distance matrix.
5. Visualize the clusters as the final step.

Data extraction and preprocessing are kept in `data_analysis_and_preprocesing.ipynb`; this notebook consumes its generated JSON. The legacy flat arrays are retained for other uses but must not be paired by index: the title and description lists have different lengths and lose their source boundaries. Instead, `attribute_records` contains unique titles with descriptions paired inside source blocks where the two lists have equal lengths. Descriptions from blocks with mismatched lengths are retained separately in `unpaired_descriptions`; they help estimate IDF weights but are never assigned to a title.


**Dependencies:** NumPy, RapidFuzz, Sentence Transformers, scikit-learn, and Plotly. They are declared in the project `pyproject.toml` and imported together in the first code cell. Use the project's Python 3.13+ environment; the first embedding run may download the configured public model.

In [1]:
# Keep every third-party and standard-library import together at the start.
import json
import re
from collections import Counter
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
from IPython.display import HTML, display
from rapidfuzz import fuzz, process, utils
from sentence_transformers import SentenceTransformer
from sklearn.cluster import AgglomerativeClustering
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.manifold import TSNE

# Keep these weights summing to one so each similarity signal has a clear contribution.
W_SEMANTIC, W_TFIDF, W_FUZZY = 0.50, 0.25, 0.25
W_TITLE = 0.50
THRESHOLD = 0.70
PENALTY = 0.30
MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"
EMBEDDING_PROMPT = "Instruct: Identify product features that describe the same property\nQuery: "
HIGHLIGHT_GROUPS = 8

c:\Users\greta\AppData\Local\venvs\baseline-clustering-py313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load aligned feature records

The preprocessor stores one `attribute_records` entry per unique title and accumulates descriptions only when title and description lists aligned within the same product or variant content block. The current catalog has 8 blocks where title and description counts differ. Their 22 descriptions are saved as `unpaired_descriptions`: they are not attached to any attribute, but are still available to the IDF calculation. This avoids both losing their vocabulary signal and inventing title-description matches.

The loader searches for the generated JSON near the notebook and workspace. It checks the required fields and reports how many attributes have descriptions. If the aligned records are missing, it stops with an explicit error instead of silently reverting to positional pairing.

In [2]:
search_roots = (Path.cwd(), *Path.cwd().parents)
candidate_paths = []
for root in search_roots:
    candidate_paths.extend(
        [
            root / "project" / "Computation-Tools-for-Data-Science---Project" / "project_cleaned" / "data_analysis_and_preprocesing" / "features_en.json",
            root / "features_en.json",
            root / "data_analysis_and_preprocesing" / "features_en.json",
            root / "project_cleaned" / "data_analysis_and_preprocesing" / "features_en.json",
        ]
    )

features_path = None
feature_data = None
for candidate in dict.fromkeys(candidate_paths):
    if candidate.is_file():
        with candidate.open(encoding="utf8") as input_file:
            candidate_data = json.load(input_file)
        if "attribute_records" in candidate_data:
            features_path = candidate
            feature_data = candidate_data
            break

if features_path is None:
    raise FileNotFoundError(
        "Could not find a features_en.json containing attribute_records. "
        "Run the preprocessing step that generates the aligned records first."
    )

attribute_records = feature_data["attribute_records"]
unpaired_descriptions = feature_data.get("unpaired_descriptions", [])
if not attribute_records:
    raise ValueError(f"No attribute records found in {features_path}")

attributes = [record["title"] for record in attribute_records]
descriptions = [record["descriptions"] for record in attribute_records]
if any(not isinstance(title, str) or not isinstance(texts, list) for title, texts in zip(attributes, descriptions)):
    raise ValueError("Each attribute record must contain a string title and a list of descriptions.")
if any(not isinstance(text, str) for text in unpaired_descriptions):
    raise ValueError("unpaired_descriptions must contain only strings.")

print(f"Loaded {len(attributes)} unique attributes from {features_path}")
if len(attributes) < 2:
    raise ValueError("At least two attributes are required for similarity-based clustering.")
print(f"Attributes with aligned descriptions: {sum(bool(texts) for texts in descriptions)}")
print(f"Unpaired descriptions retained for IDF only: {len(unpaired_descriptions)}")

Loaded 445 unique attributes from c:\Users\greta\Desktop\2_semester\comp tolls fr data science\project\Computation-Tools-for-Data-Science---Project\project_cleaned\data_analysis_and_preprocesing\features_en.json
Attributes with aligned descriptions: 433
Unpaired descriptions retained for IDF only: 22


## 2. Description-only IDF similarity

TF-IDF (term frequency–inverse document frequency) gives rare, informative words more influence than words repeated throughout the catalog. Here it is fitted on aligned description documents plus the 22 unpaired descriptions, so the complete description vocabulary informs IDF weights. Only aligned per-attribute documents are transformed into rows of the similarity matrix; unpaired descriptions never become a fake attribute row. Titles are not included in this signal. TF-IDF rows are L2-normalized by default, so their dot products form cosine similarities. Attributes without descriptions have an all-zero row here and still receive semantic and fuzzy signals in the blend.

In [3]:
description_documents = [" ".join(texts) for texts in descriptions]
idf_corpus = [document for document in description_documents if document.strip()]
idf_corpus.extend(text for text in unpaired_descriptions if text.strip())
if not idf_corpus:
    raise ValueError("No descriptions are available to calculate the IDF signal.")

tfidf = TfidfVectorizer(stop_words="english", sublinear_tf=True)
tfidf.fit(idf_corpus)
description_vectors = tfidf.transform(description_documents)
token_idf = (description_vectors @ description_vectors.T).toarray()
print(f"Description-only IDF similarity matrix: {token_idf.shape}")

Description-only IDF similarity matrix: (445, 445)


## 3. Semantic title-and-description embeddings

The sentence model converts text into numeric vectors whose direction captures contextual meaning. It encodes each unique title and each distinct aligned description once. For an attribute with descriptions, the description vectors are averaged using their occurrence counts, then blended equally with the title vector (`W_TITLE = 0.50`). This gives both the label and its observed context a voice in one vector per attribute. If no aligned description exists, the title vector is used alone. Unpaired descriptions are intentionally excluded here because they have no verified title association. The public model may need to download the first time; no access token or login is required.

In [4]:
embedding_model = SentenceTransformer(MODEL_NAME)
title_vectors = embedding_model.encode(
    attributes,
    prompt=EMBEDDING_PROMPT,
    normalize_embeddings=True,
)

all_descriptions = sorted({text for texts in descriptions for text in texts})
description_embedding_vectors = embedding_model.encode(
    all_descriptions,
    prompt=EMBEDDING_PROMPT,
    normalize_embeddings=True,
)
description_vector_by_text = {
    text: description_embedding_vectors[index]
    for index, text in enumerate(all_descriptions)
}

attribute_vectors = []
for title_vector, texts in zip(title_vectors, descriptions):
    if texts:
        description_counts = Counter(texts)
        weighted_description_vector = np.average(
            [description_vector_by_text[text] for text in description_counts],
            axis=0,
            weights=list(description_counts.values()),
        )
        vector = W_TITLE * title_vector + (1 - W_TITLE) * weighted_description_vector
        vector = vector / np.linalg.norm(vector)
    else:
        vector = title_vector
    attribute_vectors.append(vector)

attribute_vectors = np.asarray(attribute_vectors)
raw_semantic = attribute_vectors @ attribute_vectors.T
baseline_similarity = np.median(raw_semantic[np.triu_indices(len(attributes), k=1)])
if baseline_similarity >= 1:
    raise ValueError("Semantic similarities cannot be calibrated: median baseline is 1.")

# Calibrate model-specific cosine baselines to 0; identical vectors remain at 1.
semantic = np.clip(
    (raw_semantic - baseline_similarity) / (1 - baseline_similarity),
    0,
    1,
)
print(f"Semantic matrix ready; uncalibrated median baseline: {baseline_similarity:.3f}")

c:\Users\greta\AppData\Local\venvs\baseline-clustering-py313\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\greta\.cache\huggingface\hub\models--Qwen--Qwen3-Embedding-0.6B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\greta\AppData\Local\venvs\baseline-clustering-py313\Lib\site-packages\hug

Semantic matrix ready; uncalibrated median baseline: 0.568


## 4. Blend semantic, IDF, and title-string similarities

Each signal is represented as an attribute-by-attribute similarity matrix with the same row/column order as `attributes`. The weighted average combines semantic meaning (50%), description vocabulary (25%), and title wording (25%). We blend similarities rather than adding IDF values directly to sentence vectors, whose dimensions and interpretation differ. RapidFuzz's token-sort ratio is useful when wording is lexically close, including changes in token order; it complements meaning-based embeddings.

In [5]:
if not np.isclose(W_SEMANTIC + W_TFIDF + W_FUZZY, 1.0):
    raise ValueError("Similarity weights must sum to 1.")

fuzzy = process.cdist(
    attributes,
    attributes,
    scorer=fuzz.token_sort_ratio,
    processor=utils.default_process,
) / 100

similarity = (
    W_SEMANTIC * semantic
    + W_TFIDF * token_idf
    + W_FUZZY * fuzzy
)
# A missing description gives no IDF evidence, but every item has zero distance to itself.
np.fill_diagonal(similarity, 1.0)
print(f"Blended similarity matrix: {similarity.shape}")

Blended similarity matrix: (445, 445)


## 5. Apply explicit business-conflict penalties

A business constraint expresses a rule that matters to how the groups will be used. This example applies a 0.30 similarity reduction between titles that exclusively match the `winter` and `windproof` keyword patterns. If a title matches both concepts, that pair is left alone. These patterns are examples carried over from the source notebook, not automatically verified business policy; replace them with rules approved for the intended use. The clipped result remains within the 0–1 similarity range.

In [6]:
CONCEPTS = {
    "winter": r"\b(winter|cold|freez\w*|frost\w*)\b",
    "windproof": r"\b(wind\w*)\b",
}
CONFLICTS = [("winter", "windproof")]

concepts_by_attribute = [
    {
        name
        for name, pattern in CONCEPTS.items()
        if re.search(pattern, title, re.IGNORECASE)
    }
    for title in attributes
]

penalty_matrix = np.zeros_like(similarity)
for first_concept, second_concept in CONFLICTS:
    first_only = np.array(
        [first_concept in concepts and second_concept not in concepts for concepts in concepts_by_attribute]
    )
    second_only = np.array(
        [second_concept in concepts and first_concept not in concepts for concepts in concepts_by_attribute]
    )
    penalty_matrix[np.ix_(first_only, second_only)] = PENALTY
    penalty_matrix[np.ix_(second_only, first_only)] = PENALTY

similarity = np.clip(similarity - penalty_matrix, 0, 1)
print(f"Conflicting attribute pairs penalized: {int(np.count_nonzero(penalty_matrix) // 2)}")

Conflicting attribute pairs penalized: 72


## 6. Agglomerative clustering and label assignment

Agglomerative clustering starts with one cluster per attribute and merges clusters according to average pairwise distance. It accepts distances, not similarities, so the blended values are converted with `distance = 1 - similarity`. `n_clusters=None` means the number of groups is not fixed in advance; `distance_threshold = 0.70` controls when merging stops. A lower threshold generally produces more, smaller groups. `labels[i]` is the cluster assigned to `attributes[i]`; the numeric label itself has no semantic ranking.

In [7]:
distance_matrix = 1 - similarity
model = AgglomerativeClustering(
    n_clusters=None,
    metric="precomputed",
    linkage="average",
    distance_threshold=THRESHOLD,
)
labels = model.fit(distance_matrix).labels_

sizes = Counter(labels)
multi = [label for label, size in sizes.items() if size > 1]
print(f"{len(sizes)} clusters: {len(multi)} with multiple attributes, {len(sizes) - len(multi)} singletons")

# Print members so each label can be inspected before the visualization.
members_by_cluster = {
    label: [attributes[index] for index, assigned_label in enumerate(labels) if assigned_label == label]
    for label in sizes
}
for label, members in sorted(members_by_cluster.items(), key=lambda item: (-len(item[1]), item[0])):
    if len(members) > 1:
        print(f"\nCluster {label} ({len(members)} attributes)")
        for member in members:
            print(f"  - {member}")

98 clusters: 68 with multiple attributes, 30 singletons

Cluster 22 (21 attributes)
  - Insulating fleece inner
  - Naturally insulating and soft
  - Merino wool fabric mix
  - Luxurious soft merino blend
  - Microfleece fabric
  - Naturally Soft & Insulating
  - Insulating fleece inserts
  - Thermal fleece fabric
  - Fleece lining
  - Merino wool lining
  - Brushed fleece inserts
  - Pure merino wool
  - Merino wool fabric blend
  - Soft next-to-skin lining
  - No Itch Comfort
  - Next-to-skin comfort
  - Merino wool blend
  - Premium merino wool blend
  - Soft insulating liner
  - Soft brushed lining
  - Lofty lining

Cluster 87 (20 attributes)
  - Highly breathable
  - Breathable mesh back
  - Insulating & breathable
  - Breathable
  - Breathable, windproof fabric
  - Cool and comfortable
  - Breathable construction
  - Extremely breathable
  - Super breathable
  - Ideal for hot conditions
  - Thermal insulation with exceptional breathability
  - Insulating & Breathable
  - Breathab

## Clustering evaluation

_Reserved for evaluation later. No evaluation metrics or methods are calculated in this notebook._

## 7. Cluster visualization (final step)

t-SNE projects the same precomputed distances to two dimensions for display only; it does not change the clustering labels or serve as an evaluation metric. Nearby points suggest similarity in this projection, but the plot can distort some distances. The largest multi-attribute groups receive distinct colors, other groups are muted, and singleton clusters are shown as hollow markers. Hover over a point to see its attribute and group. The fixed perplexity of 30 requires more than 30 input attributes.

In [16]:
import importlib
import nbformat
import plotly.io._renderers as plotly_renderers

importlib.reload(plotly_renderers)

<module 'plotly.io._renderers' from 'c:\\Users\\greta\\AppData\\Local\\venvs\\baseline-clustering-py313\\Lib\\site-packages\\plotly\\io\\_renderers.py'>

In [17]:
from IPython.display import HTML, display

HIGHLIGHT_PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
GRAY, LIGHT_GRAY, INK, MUTED = "#8c8c86", "#c9c8c0", "#1f2933", "#6b7785"

coordinates = TSNE(
    n_components=2,
    metric="precomputed",
    init="random",
    perplexity=30,
    random_state=42,
).fit_transform(distance_matrix)


def group_name(label):
    """Use the most centrally similar attribute as the group's display name."""
    indices = np.where(labels == label)[0]
    within_group_similarity = similarity[np.ix_(indices, indices)]
    medoid_index = indices[within_group_similarity.sum(axis=1).argmax()]
    return attributes[medoid_index]


ranked_groups = sorted(multi, key=lambda label: -sizes[label])
highlighted_groups = ranked_groups[:HIGHLIGHT_GROUPS]
group_names = {label: group_name(label) for label in multi}


def make_trace(indices, name, color, hollow=False, size=9):
    return go.Scatter(
        x=coordinates[indices, 0],
        y=coordinates[indices, 1],
        mode="markers",
        name=name,
        marker=dict(
            size=size,
            color="white" if hollow else color,
            line=dict(width=1.5 if hollow else 1, color=color if hollow else "white"),
        ),
        text=[attributes[index] for index in indices],
        customdata=[group_names.get(labels[index], "Singleton") for index in indices],
        hovertemplate="<b>%{text}</b><br>group: %{customdata}<extra></extra>",
    )


other_group_indices = np.array(
    [index for index, label in enumerate(labels) if sizes[label] > 1 and label not in highlighted_groups]
)
singleton_indices = np.array([index for index, label in enumerate(labels) if sizes[label] == 1])

figure = go.Figure()
figure.add_trace(
    make_trace(singleton_indices, f"Singletons ({len(singleton_indices)})", LIGHT_GRAY, hollow=True, size=7)
)
figure.add_trace(
    make_trace(other_group_indices, f"Other groups ({len(ranked_groups) - len(highlighted_groups)})", GRAY, size=7)
)
for color, label in zip(HIGHLIGHT_PALETTE, highlighted_groups):
    indices = np.where(labels == label)[0]
    figure.add_trace(make_trace(indices, f"{group_names[label]} ({sizes[label]})", color, size=10))

figure.update_layout(
    title=dict(text="Attribute clusters: context-aware similarity (t-SNE)", x=0, font=dict(color=INK)),
    template="simple_white",
    width=1000,
    height=720,
    xaxis=dict(visible=False),
    yaxis=dict(visible=False),
    legend=dict(title="Largest groups", font=dict(color=INK), itemsizing="constant"),
    hoverlabel=dict(bgcolor="white", font_color=INK),
)
display(HTML(figure.to_html(full_html=False, include_plotlyjs="cdn")))

NameError: name 'HTML' is not defined